In [1]:
# packages
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from sklearn.model_selection import train_test_split 
from sklearn.tree import \
      (DecisionTreeRegressor as DTR, 
       DecisionTreeClassifier as DTC, 
       plot_tree)
from sklearn.ensemble import \
     (RandomForestRegressor as RFR,
      RandomForestClassifier as RFC,
      GradientBoostingRegressor as GBR)
from sklearn.model_selection import cross_val_score

# set seed
seed = 5498

Matplotlib is building the font cache; this may take a moment.


### You'll need the heart attack data (heart.csv) to complete this exercise. Your goal is to predict the risk of having a heart attack (binary).

In [2]:
heart = pd.read_csv('data/heart.csv')
heart.head()

  Patient ID  Age   Sex  Cholesterol Blood Pressure  Heart Rate  Diabetes  \
0    UEX5427   40  Male          122         144/90         110         1   
1    NBG3103   41  Male          357         174/73         109         0   
2    EIE1041   58  Male          206        107/105          63         0   
3    GYA0389   30  Male          142        132/107          86         1   
4    IXT9616   69  Male          175         153/93          98         1   

   Family History  Smoking  Obesity  ...  Sedentary Hours Per Day  Income  \
0               0        1        1  ...                 1.538008  252889   
1               1        1        1  ...                 7.030873   63196   
2               0        1        0  ...                 8.086554  107870   
3               0        1        0  ...                 5.039048  223985   
4               1        1        1  ...                 4.354972   24309   

         BMI  Triglycerides  Physical Activity Days Per Week  \
0  28.8544

In [3]:
heart.dtypes

Patient ID                             str
Age                                  int64
Sex                                    str
Cholesterol                          int64
Blood Pressure                         str
Heart Rate                           int64
Diabetes                             int64
Family History                       int64
Smoking                              int64
Obesity                              int64
Alcohol Consumption                  int64
Exercise Hours Per Week            float64
Diet                                   str
Previous Heart Problems              int64
Medication Use                       int64
Stress Level                         int64
Sedentary Hours Per Day            float64
Income                               int64
BMI                                float64
Triglycerides                        int64
Physical Activity Days Per Week      int64
Sleep Hours Per Day                  int64
Country                                str
Continent  

In [4]:
heart.shape

(6572, 26)

### Feature engineering

In [5]:
def feature_engineering(df=heart):
    df['Male'] = (df.Sex == 'Male').astype(int)
    df['HealthyDiet'] = (df.Diet == 'Healthy').astype(int)
    df['UnhealthyDiet'] = (df.Diet == 'Unhealthy').astype(int)
    df[['BP1','BP2']] = df['Blood Pressure'].str.split('/', n=1, expand=True).astype(int)

feature_engineering()

In [6]:
heart.head()

  Patient ID  Age   Sex  Cholesterol Blood Pressure  Heart Rate  Diabetes  \
0    UEX5427   40  Male          122         144/90         110         1   
1    NBG3103   41  Male          357         174/73         109         0   
2    EIE1041   58  Male          206        107/105          63         0   
3    GYA0389   30  Male          142        132/107          86         1   
4    IXT9616   69  Male          175         153/93          98         1   

   Family History  Smoking  Obesity  ...  Sleep Hours Per Day       Country  \
0               0        1        1  ...                    7      Thailand   
1               1        1        1  ...                    9        France   
2               0        1        0  ...                    5      Colombia   
3               0        1        0  ...                    9  South Africa   
4               1        1        1  ...                    7         Japan   

       Continent           Hemisphere  Heart Attack Risk  Male

In [7]:
indep_vars = ['Age', 'Male', 'Cholesterol', 'BP1', 'BP2', 'Heart Rate', 'Diabetes', 
              'Family History', 'Smoking', 'Obesity', 'Alcohol Consumption', 'HealthyDiet', 
              'UnhealthyDiet', 'Exercise Hours Per Week', 'Previous Heart Problems', 
              'Medication Use', 'Stress Level', 'Sedentary Hours Per Day', 'Income', 'BMI',
              'Triglycerides', 'Physical Activity Days Per Week', 'Sleep Hours Per Day']

X = heart[indep_vars]
y = heart['Heart Attack Risk']

In [8]:
X_train, X_test, y_train, y_test, Train, Test = train_test_split(X, y, heart, 
                                                                 random_state = seed, 
                                                                 test_size = 0.33, 
                                                                 shuffle = True)

### Univariate Analyses (Mean, Median, etc.)

In [9]:
Train.describe()

               Age  Cholesterol   Heart Rate     Diabetes  Family History  \
count  4403.000000  4403.000000  4403.000000  4403.000000     4403.000000   
mean     53.760164   261.571201    75.398592     0.656371        0.487168   
std      21.302684    81.111506    20.455754     0.474973        0.499892   
min      18.000000   120.000000    40.000000     0.000000        0.000000   
25%      35.000000   194.000000    58.000000     0.000000        0.000000   
50%      54.000000   261.000000    75.000000     1.000000        0.000000   
75%      72.000000   333.000000    93.000000     1.000000        1.000000   
max      90.000000   400.000000   110.000000     1.000000        1.000000   

           Smoking      Obesity  Alcohol Consumption  Exercise Hours Per Week  \
count  4403.000000  4403.000000          4403.000000              4403.000000   
mean      0.895526     0.503066             0.599364                10.123129   
std       0.305909     0.500047             0.490083           

### Correlation Matrix

In [10]:
X_train.corr().style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1)

### To predict heart attack risk, we'll start by building a few simple decision trees.

In [11]:
tree1 = DTC(max_depth=1)
tree1.fit(X_train, y_train)

tree2 = DTC(max_depth=2)
tree2.fit(X_train, y_train)

tree3 = DTC(max_depth=3)
tree3.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=3)

### We can draw plots of each of these decision trees

In [12]:
feature_names = list(X_train.columns)
ax = subplots(figsize=(6,6))[1]
plot_tree(tree1,
          feature_names=feature_names,
          ax=ax);

In [13]:
ax = subplots(figsize=(12,6))[1]
plot_tree(tree2,
          feature_names=feature_names,
          ax=ax);

In [14]:
ax = subplots(figsize=(18,8))[1]
plot_tree(tree3,
          feature_names=feature_names,
          ax=ax);

### Create a new tree with depth 3 that requires there to be at least 100 records in each leaf.

In [15]:
tree4 = DTC(max_depth=3, min_samples_leaf=100, random_state=seed)
tree4.fit(X_train, y_train)

ax = subplots(figsize=(18,8))[1]
plot_tree(tree4,
          feature_names=feature_names,
          ax=ax);

### Create a new version of the last tree so that it limits the number of leaf nodes to 8 rather than limiting the depth to 3.

In [16]:
tree5 = DTC(max_leaf_nodes=8, min_samples_leaf=100, random_state=seed)
tree5.fit(X_train, y_train)

ax = subplots(figsize=(18,8))[1]
plot_tree(tree5,
          feature_names=feature_names,
          ax=ax);

### Create a basic random forest model on the training set.

In [17]:
rf_basic = RFC(max_features=len(indep_vars),
               criterion='log_loss',
               n_estimators=50,
               max_depth=10,
               min_samples_leaf=1,
               bootstrap=True,
               oob_score=True,
               random_state=seed)
rf_basic.fit(X_train, y_train)

RandomForestClassifier(criterion='log_loss', max_depth=10, max_features=23,
                       n_estimators=50, oob_score=True, random_state=5498)

### We'll use this function to quickly calculate the log loss of a model.

In [18]:
def log_loss(y_true, y_prob, eps=1e-15):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    
    # avoid log(0)
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    ll = -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))
    return ll

### Compute the log loss on the training set.

In [19]:
y_hat_basic_train = rf_basic.predict_proba(X_train)[:,1]
print('train log loss:', log_loss(y_train, y_hat_basic_train))

train log loss: 0.4683745231235701


### Calculate the out-of-bag error (bootstrapping) on training data to estimate the log loss on the test set.

In [20]:
y_hat_basic_oob = rf_basic.oob_decision_function_[:,1]
print('oob log loss:  ', log_loss(y_train, y_hat_basic_oob))

oob log loss:   0.6667260171612608


### Compute the log loss on the test set.

In [21]:
y_hat_basic_test = rf_basic.predict_proba(X_test)[:,1]
print('test log loss: ', log_loss(y_test, y_hat_basic_test))

test log loss:  0.6574443298870886


### Build an improved random forest model on this same set of data. 

You will be scored according to how well it performs on an unseen holdout set.

In [22]:
# Smaller trees and larger leaves reduce overfitting.
rf_better = RFC(max_features='sqrt',
                criterion='log_loss',
                n_estimators=500,
                max_depth=3,
                min_samples_leaf=100,
                bootstrap=True,
                oob_score=True,
                random_state=seed)
rf_better.fit(X_train, y_train)

RandomForestClassifier(criterion='log_loss', max_depth=3, min_samples_leaf=100,
                       n_estimators=500, oob_score=True, random_state=5498)

### Calculate the out-of-bag log loss for this model. Store it as `log_loss_oob`

In [23]:
y_hat_better_oob = rf_better.oob_decision_function_[:, 1]

log_loss_oob = log_loss(y_train, y_hat_better_oob)
print(log_loss_oob)

0.6549883215270697


### Create the feature importance list for this model.

In [24]:
feature_imp = pd.DataFrame(
    {'importance':rf_better.feature_importances_},
    index=indep_vars)
feature_imp

                                 importance
Age                                0.061720
Male                               0.003556
Cholesterol                        0.062774
BP1                                0.074610
BP2                                0.062417
Heart Rate                         0.054744
Diabetes                           0.009054
Family History                     0.007940
Smoking                            0.001505
Obesity                            0.010957
Alcohol Consumption                0.012369
HealthyDiet                        0.002535
UnhealthyDiet                      0.006504
Exercise Hours Per Week            0.078445
Previous Heart Problems            0.005212
Medication Use                     0.005265
Stress Level                       0.032737
Sedentary Hours Per Day            0.117860
Income                             0.082437
BMI                                0.082479
Triglycerides                      0.101625
Physical Activity Days Per Week 

### Sort the importance values and then create a horizontal bar chart that displays the sorted variable importances.

In [25]:
feature_imp_sorted = feature_imp.sort_values('importance')

# Draw a horizontal bar chart
feature_imp_sorted.plot.barh(figsize=(8, 7), legend=False)
plt.xlabel('Feature importance')
plt.ylabel('Feature')
plt.title('Random forest feature importances')
plt.tight_layout()
plt.show()

## Discussion Questions

### (1) Explain what "value" means in the decision tree plots.

"value" shows the number of observations in each class at a node, in the order [0, 1]: no heart attack risk, then heart attack risk. The class with the larger count is the node's prediction. At a leaf, the fraction in class 1 gives the predicted probability of heart attack risk.

### (2) How is “log loss” different from the log likelihood we defined in class?

Log loss is the negative average log likelihood: log loss = -log likelihood / n. We minimize log loss, while we maximize log likelihood. Both favor assigning higher probabilities to the outcomes that actually occur.

### (3) Explain how we know the basic model is overfit before attempting to build any other models.

The basic model has a training log loss of about 0.468, compared with 0.667 out of bag and 0.657 on the test set. The higher losses on unseen observations show that it fits the training data better than it generalizes, which is evidence of overfitting.